In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

lyrics = pd.read_csv("lyrics_en_es.csv")
main = pd.read_csv("Spotify_weeklytop200_cleaned.csv")

song_stage = main[["id", "Stage"]]  

df = lyrics.merge(song_stage, on="id", how="inner")
print(df.groupby("Stage")["id"].count())

Stage
Stage 1 (2017-2019)    27303
Stage 2 (2020-2022)    27970
Stage 3 (2023-2025)    26488
Name: id, dtype: int64


In [13]:
def clean_lyrics(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'\(.*?\)', ' ', text)
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)
df.head(3)[["Title", "Stage", "lyrics_clean"]]

,Title,Stage,lyrics_clean
0,Starboy,Stage 1 (2017-2019),ayy i m tryna put you in the worst mood ah p c...
1,Starboy,Stage 1 (2017-2019),ayy i m tryna put you in the worst mood ah p c...
2,Starboy,Stage 1 (2017-2019),ayy i m tryna put you in the worst mood ah p c...


In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import nltk
from nltk.corpus import stopwords

In [14]:
custom_stops_en = list(ENGLISH_STOP_WORDS) + [
    # Contraction fragments (since apostrophes removed during cleaning)
    "ve", "ain", "ft", "re", "ll", "don", "didn", "doesn",
    "couldn", "wouldn", "shouldn", "wasn", "weren", "isn", "aren",
    # Lyric filler & informal words (1st refinement)
    "yeah", "oh", "just", "ooh", "la", "la la", "oh oh", "uh",
    "ah", "na", "da", "hey", "nah", "aye", "ayy", "woah", "whoa",
    "hm", "mm", "wa", "eh", "feat", "like", "got", "let", "say",
    "wanna", "gonna", "gotta", "cause", "cuz", "em", "til", "tryna",
    "come", "make", "see", "back", "take", "tell", "go", "think",
    "away", "right", "good", "man", "stay", "never", "get", "know","lil", "la", "yo"
]

In [15]:
spanish_stops = set(stopwords.words('spanish'))
custom_stops = list(spanish_stops)+ [
    # Spanish function words
    "que", "la", "el", "en", "de", "los", "las", "un", "una", "por",
    "con", "su", "para", "es", "al", "del", "lo", "le", "se", "me",
    "te", "si", "pero", "más", "como", "ya", "todo", "esta", "yo",
    "mi", "tu", "hay", "fue", "son", "muy", "bien", "cuando", "tiene",
    "ser", "era", "también", "hasta", "desde", "sobre", "sin", "entre",
    "ni", "solo", "tan", "ese", "esa", "aquí", "tú", "usted", "porque",
    "donde", "después", "antes", "algo", "nada", "todos", "otro",
    # Lyric filler words (1st refinement)
    "yeah", "oh", "ooh", "ah", "na", "da", "hey", "nah", "aye", "ayy",
    "woah", "whoa", "uh", "hm", "mm", "gonna", "wanna", "gotta",
    "cause", "cuz", "em", "til", "tryna", "ll", "don", "got",
    "let", "just", "like", "know", "baby", "love", "feat", "wa", "eh",
    "yo", "la",
    # Spanish filler & contracted forms(2nd refinement)
    "pa", "ey", "yeh", "oi", "oi oi", "ve", "ti", "va", "nos", "voy", "soy"
]

In [16]:
stages = ["Stage 1 (2017-2019)", "Stage 2 (2020-2022)", "Stage 3 (2023-2025)"]

def get_stage_docs(lang):
    return [
        " ".join(df[(df["Stage"] == s) & (df["language"] == lang)]["lyrics_clean"])
        for s in stages
    ]

In [17]:
stage_docs_en = get_stage_docs("en")

tfidf_en = TfidfVectorizer(stop_words=custom_stops_en, max_features=5000, ngram_range=(1, 2), min_df=1)
tfidf_matrix_en = tfidf_en.fit_transform(stage_docs_en)

tfidf_df_en = pd.DataFrame(
    tfidf_matrix_en.toarray(),
    columns=tfidf_en.get_feature_names_out(),
    index=stages
)

for stage in stages:
    print(f"\n{stage}:")
    print(tfidf_df_en.loc[stage].nlargest(15).to_string())


Stage 1 (2017-2019):
love     0.435839
baby     0.244609
want     0.210322
time     0.193545
need     0.151872
way      0.149440
feel     0.148316
bitch    0.119607
girl     0.112740
fuck     0.109545
look     0.107223
bad      0.103625
life     0.103356
mind     0.103246
shit     0.099311

Stage 2 (2020-2022):
love     0.431175
baby     0.240687
time     0.200787
want     0.185106
way      0.153572
feel     0.151526
need     0.149344
said     0.122525
day      0.121520
life     0.119691
heart    0.118507
girl     0.116341
night    0.109902
look     0.101649
bad      0.101087

Stage 3 (2023-2025):
love     0.412769
baby     0.279943
time     0.235539
want     0.179479
way      0.163236
feel     0.153110
night    0.140662
said     0.130462
life     0.129147
need     0.128031
day      0.108433
look     0.097084
heart    0.094100
world    0.092951
mind     0.091355


In [18]:
stage_docs_es = get_stage_docs("es")

tfidf_es = TfidfVectorizer(stop_words=custom_stops, max_features=5000, ngram_range=(1, 2), min_df=1)
tfidf_matrix_es = tfidf_es.fit_transform(stage_docs_es)

tfidf_df_es = pd.DataFrame(
    tfidf_matrix_es.toarray(),
    columns=tfidf_es.get_feature_names_out(),
    index=stages
)

for stage in stages:
    print(f"\n{stage}:")
    print(tfidf_df_es.loc[stage].nlargest(15).to_string())


Stage 1 (2017-2019):
quiero     0.306836
sé         0.232474
quiere     0.204368
así        0.177400
ahora      0.149805
conmigo    0.147615
noche      0.147359
amor       0.133306
siempre    0.126962
to         0.122809
bebé       0.115071
cómo       0.114616
vamo       0.113819
dura       0.110747
nadie      0.109324

Stage 2 (2020-2022):
quiero     0.275475
to         0.225575
sé         0.197553
quiere     0.188869
siempre    0.186588
noche      0.164969
ahora      0.158644
hace       0.146460
cómo       0.143712
ere        0.136195
dime       0.134977
contigo    0.131503
hoy        0.119190
así        0.112502
sabe       0.111076

Stage 3 (2023-2025):
quiero    0.265962
sé        0.217339
mami      0.152776
ahora     0.152462
noche     0.150439
así       0.143812
to        0.141021
bebé      0.140916
hoy       0.138719
cómo      0.136626
kuduro    0.136423
quiere    0.130243
amor      0.127871
ver       0.114721
vez       0.111303
